# Training (discrete 4-quadrant): topographic `BioLeakyRNNTopo`

Same curriculum as `01c_train_topo.ipynb` but with
`continuous_locations=False` — target/cue restricted to the 4 fixed
STIM_POS quadrants. This produces clean spatial conditions per CTOA bin
(matching Amengual's 4-position monkey paradigm) so that
tangling/dimensionality analyses (notebook `11`) get an inverted-U
structure rather than rotational-averaging cancellation.

**Checkpoints** are saved separately as `stage{0,1,2,3}_topo_discrete.pt`
so the continuous-trained `_topo` weights remain untouched.


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.model_topo import BioLeakyRNNTopo
from src.plotting_topo import plot_rf_drive, plot_sheet_layout, plot_fix_weights
from src.env import CuedTargetWithDistractorsV3
from src.training import TrainConfig, train_supervised

device = "cpu"
print("device:", device)
Path("checkpoints").mkdir(exist_ok=True)

from src.figsave import autosave
autosave("01d_train_topo_discrete")


## Ablation: stim_on and cue_on removed

Training uses `NoTimingMarkersEnv` — a subclass of `CuedTargetWithDistractorsV3`
that zeros **both** the `cue_on` (channel 1) and `stim_on` (channel 4)
channels in every trial. This removes two shortcut signals that earlier
trainings exploited:
- `stim_on` was used as a pure timing marker for target detection
  (original ablation in the previous iteration).
- `cue_on` was used as a pure timing marker for the cue epoch — lesion
  tests showed the network relied on it (zeroing it dropped accuracy
  0.80 → 0.39) without actually decoding *where* the cue pointed
  (zeroing `cue_x/cue_y` left accuracy at 0.81).

With both markers absent, the only way to detect either the cue epoch
or the target is via non-zero `cue_x/cue_y` or `stim_x/stim_y` drive
through the frozen topographic `W_in_topo`, so the network is forced to
use spatial structure for both events.

The environment source (`src/env.py`) is untouched — the ablation is a
notebook-only subclass.


In [ ]:
def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=True,
        tau_adapt=100.0,
        g_adapt=0.5,
    ).to(device)

In [ ]:
# Sanity: visualize sheet (E integer grid + I random points) and a sample
# RF drive at stim=(1, 1) to confirm geometric input path works.
m0 = make_model()
plot_sheet_layout(m0)
plt.show()
plot_rf_drive(m0, stim_xy=(-0.7, 0.7), strength=1.0)
plt.show()
plot_rf_drive(
    m0, stim_xy=(-0.4, 0.4), strength=1.0, title="RF drive @ cue loc 1 (-0.4, 0.4)"
)
plt.show()
del m0

## Stage 0 — detect target, no cue, no distractors

In [ ]:
def make_env_stage0():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=0.0,
        p_distractor_trial=0.0,
        distractor_strength=0.0,
        continuous_locations=False,
    )


model = make_model()
cfg0 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
    fa_penalty_weight=0.0,
    com_weight=0.5,
    sparsity_weight=0.01,
    stop_on_no_miss=0,
)
history0 = train_supervised(model, make_env_stage0, cfg0)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage0_topo_discrete.pt")
print("Saved: checkpoints/stage0_topo_discrete.pt")

## Stage 1 — add cue

In [ ]:
def make_env_stage1():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.0,
        distractor_strength=0.0,
        continuous_locations=False,
    )


model = make_model()
model.load_state_dict(
    torch.load("checkpoints/stage0_topo_discrete.pt", weights_only=True)["state_dict"],
    strict=False,
)
cfg1 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=1000,
    print_every=100,
    device=device,
    fa_penalty_weight=0.0,
    com_weight=0.5,
    sparsity_weight=0.01,
    stop_on_no_miss=0,
)
history1 = train_supervised(model, make_env_stage1, cfg1)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage1_topo_discrete.pt")
print("Saved: checkpoints/stage1_topo_discrete.pt")

## Stage 2 — cue + distractors (with early stopping)

In [ ]:
def make_env_stage2():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=False,
    )


model = make_model()
model.load_state_dict(
    torch.load("checkpoints/stage1_topo_discrete.pt", weights_only=True)["state_dict"],
    strict=False,
)
cfg2 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=15000,
    print_every=50,
    device=device,
    fa_penalty_weight=0.0,
    com_weight=0.5,
    sparsity_weight=0.01,
)  # early stopping enabled (default stop_on_no_miss=3)
history2 = train_supervised(model, make_env_stage2, cfg2)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage2_topo_discrete.pt")
print("Saved: checkpoints/stage2_topo_discrete.pt")

## Training curves

In [ ]:
def smooth(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, hist, stage in zip(axes, [history0, history1, history2], [0, 1, 2]):
    updates = np.arange(1, len(hist["p_correct"]) + 1)
    sw = 20
    for key, color, label in [
        ("p_correct", "green", "correct"),
        ("p_miss", "orange", "miss"),
        ("p_abort", "red", "abort"),
    ]:
        vals = hist[key]
        ax.plot(updates, vals, alpha=0.15, color=color, lw=0.8)
        if len(vals) >= sw:
            ax.plot(updates[sw - 1 :], smooth(vals, sw), color=color, lw=2, label=label)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Stage {stage} (topo)")
    ax.set_xlabel("update")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, hist, stage in zip(axes, [history0, history1, history2], [0, 1, 2]):
    updates = np.arange(1, len(hist["loss"]) + 1)
    sw = 20
    ax.plot(updates, hist["loss"], alpha=0.15, color="steelblue", lw=0.8)
    if len(hist["loss"]) >= sw:
        ax.plot(
            updates[sw - 1 :],
            smooth(hist["loss"], sw),
            color="steelblue",
            lw=2,
            label="total",
        )
    ax.plot(updates, hist["ce"], alpha=0.15, color="tomato", lw=0.8)
    if len(hist["ce"]) >= sw:
        ax.plot(
            updates[sw - 1 :], smooth(hist["ce"], sw), color="tomato", lw=2, label="CE"
        )
    ax.set_title(f"Stage {stage} loss (topo)")
    ax.set_xlabel("update")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Stage 3 — long fine-tune for tangling-vs-training analysis

Continues from `stage2_topo_discrete.pt` for 10 000 updates with **early stopping disabled**
(we don't care about p_miss — we want to see how the population dynamics keep
evolving once the behaviour is already correct). Final weights go to
`stage3_topo_discrete.pt`. Together with `stage{0,1,2}_topo.pt` you have 4 snapshots
(end of each curriculum stage) — perfect for plotting tangling and effective
dimensionality vs training stage.


In [ ]:
model = make_model()
model.load_state_dict(
    torch.load("checkpoints/stage2_topo_discrete.pt", weights_only=True)["state_dict"],
    strict=False,
)

cfg3 = TrainConfig(
    batch_size=64,
    lr=1e-3,
    max_updates=10000,
    print_every=200,
    device=device,
    fa_penalty_weight=0.0,
    com_weight=0.5,
    sparsity_weight=0.01,
    stop_on_no_miss=0,                # no early stop
)
history3 = train_supervised(model, make_env_stage2, cfg3)
torch.save({"state_dict": model.state_dict()}, "checkpoints/stage3_topo_discrete.pt")
print("Saved: checkpoints/stage3_topo_discrete.pt")
